In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/esic",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import optuna
from optuna.trial import Trial
import joblib
import random

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.esic_v1 import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.collection_utils import SafetyDict
from sj_utils.evaluator import TimeChecker

In [ ]:
from util import get_token_saver_loader_transcriber, normalize_text

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/dev"
STORAGE = "/workspaces/dev/storage/esic/"
STUDY = "/workspaces/dev/study/esic/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)
storage = Path(STORAGE)
study_path = Path(STUDY) / "20250721_study_3.pkl"
study_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
src_folder = search_all_data(src)
len(src_folder)

In [ ]:
transcriber = get_token_saver_loader_transcriber(src, storage, SAMPLE_RATE)

In [ ]:
def objective(trial:Trial):
    hyperparameters = SafetyDict({
        "whisper": {
            "model_options": {
                "model_size_or_path": "large-v3",
                "device": "cuda",
                "compute_type": "float16",
            },
            "transcribe_options": {
                "beam_size":5,
                "vad_filter": False,
                "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
            }
        },
        "silero_vad": {
            "model_options": {},
            "run_options": {}
        },
        "asr": {
            "max_overlap_duration": trial.suggest_int(
                "overlap_duration", 16000, 112000, step=16000
            )
        },
        "position_weighted_filter": {
            "boundary":trial.suggest_int(
                "boundary", 0, 32000, step=100
            )
        },
        "duration_filter":{
            "z_thresh":{
                "default": 2.0,
                "en": trial.suggest_float("df_z_thresh", 0, 10.0, step = 0.1)
            },
            "min_dur": {
                "default": 160,
                # "en": 160
                "en": trial.suggest_float("df_min_duration", 0, 8000, step = 160)
            },
        },
        "probability_filter":{
            "z_thresh":{
                "default": 3.0,
                "en": trial.suggest_float("pf_z_thresh", 0.0, 10.0, step = 0.1)
            },
            "min_prob": {
                "default": 1.0,
                "en": trial.suggest_float("pf_min_prob", 0, 1, step = 0.01)
            },
        },
        "selector":{
            "iou_threshold": {
                "default": 0.5,
                # "en": trial.suggest_float("iou_threshold", 0.7, 0.91, step=0.01)
                "en": trial.suggest_float("iou_threshold", 0, 1, step=0.01)
            },
            "cos_threshold":{
                "default": 0.5,
                # "en": trial.suggest_float("cos_threshold", 0.2, 0.8, step=0.01)
                "en": trial.suggest_float("cos_threshold", 0, 1, step=0.01)
            },
            "padding": {
                "default": 3200,
                # "en": 3200
                "en": trial.suggest_int("padding", 0, 32000, step=100)
            },
        },
    })

    samples = random.sample(src_folder, 151)

    data = {}
    for sample in samples:
        trans_txt = search_file_from_dir(sample, "o")
        ref = trans_txt_to_sclite_trn(trans_txt, normalize_text)
        pred = transcriber(
            search_file_from_dir(sample, "mp4"),
            TimeChecker(),
            hyperparameters,
            hyperparameters["asr"]["max_overlap_duration"]
        )
        hyp = TRNFormat(id = ref.id, text = pred)
        data[f"{sample.parent.stem}_{sample.stem}"] = {"ref": ref,"hyp": hyp}

    concat_result = {}
    for value in data.values():
        for k, v in value.items():
            if k not in concat_result:
                concat_result[k] = []
            concat_result[k].append(v)

    output = sclite_trn(concat_result["ref"], concat_result["hyp"])
    result = parse_sclite_summary(output)

    return result["wer_percent"]

In [ ]:
if study_path.exists():
    study = joblib.load(study_path)
else:
    study = optuna.create_study(direction="minimize")

In [ ]:
for _ in range(500):
    study.optimize(objective, n_trials=5)
    joblib.dump(study, study_path)

In [ ]:
study.best_value

In [ ]:
study.best_params